# Imports & Setup

In [25]:
import tomllib
from pathlib import Path
import polars as pl

In [26]:
config_file_path = Path("config.toml")

with config_file_path.open("rb") as config_file:
    config = tomllib.load(config_file)

COMPANIES = config['companies']

# Data Cleaning

In [27]:
companies = pl.read_parquet('ETL/data/raw/parquet/companies.parquet')
share_prices = pl.read_parquet('ETL/data/raw/parquet/share_prices.parquet')

In [28]:
with pl.Config(tbl_rows=25):
    plausible_companies = companies.drop_nulls(subset=pl.col('Number Employees')).sort('Number Employees', descending=True)
    selected_companies = plausible_companies.filter(pl.col('Company Name').is_in(COMPANIES))
    display(selected_companies)

SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
i64,str,f64,str,f64,f64,str,str,f64,str
62747,"""AMAZON COM INC""",103002.0,"""US0231351067""",12.0,1.298e6,"""Amazon.com Inc is an online re…","""us""",1.018724e6,"""USD"""


In [29]:
SIMFIN_IDS = [id for id in selected_companies['SimFinId']]

In [30]:
with pl.Config(tbl_rows=25):
    share_prices = share_prices.filter(pl.col('SimFinId').is_in(SIMFIN_IDS))
    display(share_prices)

SimFinId,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding
i64,f64,f64,f64,f64,f64,i64,f64,f64
62747,100.86,101.79,99.88,100.58,100.58,102279660,null,9.9800e9
62747,101.05,102.2,100.56,102.15,102.15,79546260,null,9.9800e9
62747,102.22,102.65,100.88,102.14,102.14,93112340,null,9.9800e9
62747,102.0,109.0,101.9,108.44,108.44,134334180,null,9.9800e9
62747,110.02,114.6,109.31,114.17,114.17,161743860,null,9.9800e9
62747,112.88,116.67,112.25,115.38,115.38,137331340,null,9.9800e9
62747,117.3,123.05,116.75,120.41,120.41,240764020,null,9.9800e9
62747,118.62,120.0,115.8,118.75,118.75,158600200,null,9.9800e9
62747,119.5,122.25,119.3,119.68,119.68,115413880,null,9.9800e9


In [31]:
share_prices = share_prices.drop(pl.col('Dividend'), pl.col('SimFinId'))
share_prices

Open,High,Low,Close,Adj. Close,Volume,Shares Outstanding
f64,f64,f64,f64,f64,i64,f64
100.86,101.79,99.88,100.58,100.58,102279660,9.9800e9
101.05,102.2,100.56,102.15,102.15,79546260,9.9800e9
102.22,102.65,100.88,102.14,102.14,93112340,9.9800e9
102.0,109.0,101.9,108.44,108.44,134334180,9.9800e9
110.02,114.6,109.31,114.17,114.17,161743860,9.9800e9
…,…,…,…,…,…,…
204.8,209.98,203.26,208.36,208.36,38610085,1.0598e10
204.4,205.77,198.3,200.7,200.7,49863755,1.0598e10
199.49,202.27,192.53,199.25,199.25,59802821,1.0598e10
